**Calling Library**

In [ ]:
import os
import cv2
import shutil
import random
import kagglehub
import numpy as np
import pandas as pd
from PIL import Image
import seaborn as sns
import tensorflow as tf
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from tensorflow.keras.preprocessing import image
from tensorflow.keras import layers, models,callbacks ,optimizers
from tensorflow.keras.callbacks import  EarlyStopping , ModelCheckpoint ,ReduceLROnPlateau
from tensorflow.keras.applications import MobileNetV2

**Download Data**

In [ ]:
path = kagglehub.dataset_download("snehprajapati/sign-language-detection-words")
print("Path to dataset files:", path)

# **Split of data train and val and test in folder**

In [ ]:
train_ratio = 0.7
val_ratio = 0.15
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
source_path = "/kaggle/input/sign-language-detection-words/ASL_training_data_final(1)"
path = "/content/ASL_training_data_final"

if os.path.exists(path):
    shutil.rmtree(path)
shutil.copytree(source_path,path)

classes = [c for c in os.listdir(path) if os.path.isdir(os.path.join(path, c))]
for class_name in classes:

    class_path = os.path.join(path, class_name)

    images = os.listdir(class_path)
    random.shuffle(images)

    total_images = len(images)

    train_split = int(train_ratio * total_images)
    val_split = int(val_ratio * total_images)

    train_images = images[:train_split]
    val_images = images[train_split:train_split + val_split]
    test_images = images[train_split + val_split:]

    os.makedirs(os.path.join(path, "train", class_name), exist_ok=True)
    os.makedirs(os.path.join(path, "val", class_name), exist_ok=True)
    os.makedirs(os.path.join(path, "test", class_name), exist_ok=True)

    for image in train_images:
        shutil.copy(os.path.join(class_path, image),os.path.join(path, "train", class_name, image))

    for image in val_images:
        shutil.copy(os.path.join(class_path, image),os.path.join(path, "val", class_name, image))

    for image in test_images:
        shutil.copy(os.path.join(class_path, image),os.path.join(path, "test", class_name, image))

In [ ]:
print(os.listdir("/content/ASL_training_data_final/train"))

# **Create DataFrame train data**

In [ ]:
path_train="/content/ASL_training_data_final/train"
list_file_train=[]
for folder_image in os.listdir(path_train):
    folders_train=os.path.join(path_train,folder_image)
    for training_image in os.listdir(folders_train):
      if training_image.endswith((".jpg")):
        list_file_train.append([os.path.join(folders_train,training_image),folder_image])

In [ ]:
df_train=pd.DataFrame(list_file_train,columns=["path","label"])
df_train.head()

In [ ]:
print("Length",len(list_file_train))
print(list_file_train[:5])

In [ ]:
print(len(df_train['label'].unique()))
print(df_train['label'].value_counts())

# **Create DataFrame validation data**

In [ ]:
path_val="/content/ASL_training_data_final/val"
list_file_val=[]
for folder_image in os.listdir(path_val):
    folders_val=os.path.join(path_val,folder_image)
    for val_image in os.listdir(folders_val):
      if val_image.endswith((".jpg")):
        list_file_val.append([os.path.join(folders_val,val_image),folder_image])


In [ ]:
df_val=pd.DataFrame(list_file_val,columns=["path","label"])
df_val.head()

In [ ]:
print("Length",len(list_file_val))
print(list_file_val[:5])

In [ ]:
print(len(df_val['label'].unique()))
print(df_val['label'].value_counts())

# **Create DataFrame test data**

In [ ]:
path_test="/content/ASL_training_data_final/test"
list_file_test=[]
for folder_image in os.listdir(path_test):
    folders_test=os.path.join(path_test,folder_image)
    for testing_image in os.listdir(folders_test):
      if testing_image.endswith((".jpg")):
        list_file_test.append([os.path.join(folders_test,testing_image),folder_image])

In [ ]:
df_test=pd.DataFrame(list_file_test,columns=["path","label"])
df_test

In [ ]:
print("Length",len(list_file_test))
print(list_file_test[:5])

In [ ]:
print(len(df_test['label'].unique()))
print(df_test['label'].value_counts())

# **Check size in image data**

In [ ]:
sizes = set()
for item in list_file_train:
    path = item[0]
    img = Image.open(path)
    sizes.add(img.size)
print(sizes)

In [ ]:
sizes = set()
for item in list_file_val:
    path = item[0]
    img = Image.open(path)
    sizes.add(img.size)
print(sizes)

In [ ]:
sizes = set()
for item in list_file_test:
    path = item[0]
    img = Image.open(path)
    sizes.add(img.size)
print(sizes)

# **Split data in Feature and Target and Encoder**

In [ ]:
x_train=df_train['path']
y_train=df_train['label']

x_val=df_val['path']
y_val=df_val['label']

x_test=df_test['path']
y_test=df_test['label']

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_val   = le.transform(y_val)
y_test  = le.transform(y_test)

# **model MobileNetV2**

**Preprocessing image**

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

IMG_SIZE = (224, 224)

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    img = preprocess_input(img)
    return img, label

In [ ]:
train_ds = tf.data.Dataset.from_tensor_slices((x_train.values, y_train))
val_ds = tf.data.Dataset.from_tensor_slices((x_val.values, y_val))
test_ds = tf.data.Dataset.from_tensor_slices((x_test.values, y_test))

train_ds = train_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).shuffle(1000).batch(32).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)

**Bulid base model MobileNetV2**

In [ ]:
IMG_SIZE = (224, 224, 3)
base_model_MobileNetV2 = MobileNetV2(weights="imagenet",include_top=False,input_shape=IMG_SIZE)

base_model_MobileNetV2.trainable = False

model_MobileNetV2 = models.Sequential([base_model_MobileNetV2,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(22, activation="softmax")])

model_MobileNetV2.summary()

**Optimizer and Callbacks function in model**

In [ ]:
ADAM=tf.keras.optimizers.Adam(learning_rate=0.001)
model_MobileNetV2.compile(optimizer=ADAM,loss="sparse_categorical_crossentropy",metrics=['accuracy'])
callbacks = [EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True,verbose=1),
             ReduceLROnPlateau(monitor="val_loss", factor=0.4, patience=2,verbose=1),
             ModelCheckpoint(filepath="best_model_MobileNet.keras", monitor="val_loss",verbose=1,save_best_only=True)]

**Fitted model MobileNetV2**

In [ ]:
history_mobileNetV2 = model_MobileNetV2.fit(train_ds,validation_data=val_ds,epochs=20,callbacks=callbacks,verbose=1)

**Evaluate model MobileNetV2**


In [ ]:
test_loss, test_acc = model_MobileNetV2.evaluate(test_ds)
print("Test Accuracy:", test_acc)

**Test model MobileNetV2**

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
IMG_SIZE = (224,224)
img_path = "/content/WIN_20260313_05_04_58_Pro.jpg"
# تحميل الصورة
img = tf.keras.preprocessing.image.load_img(img_path, target_size=IMG_SIZE)
img_array = tf.keras.preprocessing.image.img_to_array(img)

img_array = np.expand_dims(img_array, axis=0)
img_array = preprocess_input(img_array)

# prediction
pred = model_MobileNetV2.predict(img_array)

# Get class names from the LabelEncoder
class_names = le.classes_

# استخراج أعلى 3
top3_indices = np.argsort(pred[0])[-3:][::-1]

print("Top 3 Predictions:\n")

for i in top3_indices:
    print(f"{class_names[i]} : {pred[0][i]*100:.2f}%")

# عرض الصورة
plt.imshow(img)
plt.axis("off")
plt.title(f"Prediction: {class_names[top3_indices[0]]}")
plt.show()

# **Plot Training History With Plotly**

In [ ]:
def plot_training_history_with_plotly(history):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=list(range(1, len(history.history['accuracy']) + 1)),
        y=history.history['accuracy'],
        mode='lines',
        name='Train Accuracy'
    ))

    fig.add_trace(go.Scatter(
        x=list(range(1, len(history.history['val_accuracy']) + 1)),
        y=history.history['val_accuracy'],
        mode='lines',
        name='Validation Accuracy'
    ))

    fig.update_layout(
        title='Model Accuracy',
        xaxis_title='Epoch',
        yaxis_title='Accuracy',
        legend=dict(x=0, y=1),
    )

    fig.show()

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=list(range(1, len(history.history['loss']) + 1)),
        y=history.history['loss'],
        mode='lines',
        name='Train Loss'
    ))

    fig.add_trace(go.Scatter(
        x=list(range(1, len(history.history['val_loss']) + 1)),
        y=history.history['val_loss'],
        mode='lines',
        name='Validation Loss'
    ))

    fig.update_layout(
        title='Model Loss',
        xaxis_title='Epoch',
        yaxis_title='Loss',
        legend=dict(x=0, y=1),
    )

    fig.show()

In [ ]:
plot_training_history_with_plotly(history_mobileNetV2)